# Variational Autoencoder (VAE)
Variational Autoencoder, which builds upon standard autoencoders by introducing a **probabilistic framework** that enables:

- **Smooth latent spaces** for better interpolation
- **Generative capabilities** to create new samples
- **Principled regularization** through KL divergence
- **Theoretical foundations** in variational inference

**Prerequisites:**
- Understanding of standard autoencoders
- Basic probability (Gaussian distributions)
- PyTorch fundamentals

**What You'll Learn:**
1. Probabilistic framework and ELBO
2. Reparameterization trick for gradient flow
3. VAE implementation from scratch
4. Latent space exploration and generation
5. Comparison with standard autoencoders
6. Hands-on exercises for deeper understanding

## 1. Introduction & VAE Theory

### 1.1 Why VAEs? Limitations of Standard Autoencoders

Standard autoencoders have several limitations:

**Problem 1: Irregular Latent Space**
- No guarantee that nearby points decode to similar images
- Gaps and discontinuities in latent space
- Interpolation between points may produce nonsense

**Problem 2: Poor Generation**
- Cannot reliably sample new points from latent space
- No probabilistic interpretation
- Deterministic encoding

**Problem 3: No Regularization**
- Latent space structure is uncontrolled
- May overfit to training data
- No smooth manifold

**VAEs solve these problems** by:
- Imposing a probabilistic structure on the latent space
- Regularizing with KL divergence
- Creating smooth, continuous latent representations

### 1.2 Probabilistic Framework

VAEs are **generative models** that learn the data distribution p(x).

**Key Idea:**
- Assume data x is generated from latent variable z
- Latent variable z ~ p(z) = N(0, I) (standard normal prior)
- Generation process: sample z, then generate x from p(x|z)

**The Model:**

```
Prior:       p(z) = N(0, I)
Likelihood:  p(x|z) = decoder(z)
Marginal:    p(x) = ∫ p(x|z)p(z)dz  (intractable!)
```

**The Challenge:**
- We want to maximize log p(x) for our data
- But p(x) requires integrating over all possible z (intractable)
- Solution: Use variational inference

**Variational Inference:**
- Introduce approximate posterior q(z|x) ≈ p(z|x)
- q(z|x) = encoder(x) = N(μ(x), σ²(x))
- Encoder outputs mean μ and variance σ² for each input x

### 1.3 Evidence Lower Bound (ELBO)

Since log p(x) is intractable, we maximize a lower bound instead.

**Mathematical Derivation:**

The Evidence Lower Bound (ELBO) is:

$$
\log p(x) \geq \mathbb{E}_{q(z|x)}[\log p(x|z)] - \text{KL}(q(z|x) \| p(z))
$$

This is called ELBO because it's a lower bound on the log evidence log p(x).

**In Practice (Loss Function):**

We want to MAXIMIZE the ELBO, which is equivalent to MINIMIZING:

$$
\mathcal{L} = -\mathbb{E}_{q(z|x)}[\log p(x|z)] + \text{KL}(q(z|x) \| p(z))
$$

$$
\mathcal{L} = \text{Reconstruction Loss} + \text{KL Divergence}
$$

**Two Components:**

1. **Reconstruction Loss**: $-\mathbb{E}_{q(z|x)}[\log p(x|z)]$
   - How well we reconstruct the input
   - In practice: Binary Cross-Entropy (BCE) or Mean Squared Error (MSE)
   - Encourages accurate reconstructions

2. **KL Divergence**: $\text{KL}(q(z|x) \| p(z))$
   - How close q(z|x) is to prior p(z) = N(0, I)
   - Regularization term
   - Prevents overfitting and ensures smooth latent space
   - Has closed form for Gaussians!

**Closed Form KL Divergence:**

For q(z|x) = N(μ, σ²) and p(z) = N(0, I):

$$
\text{KL}(q(z|x) \| p(z)) = -\frac{1}{2} \sum_{j=1}^{J} (1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2)
$$

where J is the latent dimension.

### 1.4 The Reparameterization Trick

**Problem: Cannot Backpropagate Through Sampling**

When we sample z ~ N(μ, σ²), this is a **stochastic operation**. The issue:

- Sampling involves randomness (drawing from a distribution)
- Random operations have **no gradient**
- If z = sample(μ, σ²), we cannot compute ∂z/∂μ or ∂z/∂σ
- This **breaks backpropagation** - gradients cannot flow back to encoder parameters

**Why No Gradient?**

The sampling operation is:
```python
z = torch.normal(mu, sigma)  # Different each time
```

The gradient ∂z/∂μ is undefined because z changes randomly even if μ stays the same.

**Solution: Reparameterization Trick**

Instead of sampling z directly, we **reparameterize**:

1. Sample ε ~ N(0, 1) (standard normal, **independent of parameters**)
2. Compute z = μ + σ ⊙ ε (**deterministic transformation**)

**Why This Works:**

Now gradients can flow:
- ∂z/∂μ = 1 ✓
- ∂z/∂σ = ε ✓

The randomness is moved to ε (which doesn't need gradients), while μ and σ are deterministic functions of the input!
This allows backpropagation to flow through the sampling operation.
Backpropagation gets calculated for the deterministic transformation z = μ + σ ⊙ ε.

**Note:** We output log(σ²) instead of σ² for numerical stability.

### 1.5 Architecture Comparison

| Component | Standard Autoencoder | Variational Autoencoder |
|-----------|---------------------|------------------------|
| **Encoder Output** | Single vector z | Mean μ and log-variance log(σ²) |
| **Bottleneck** | Deterministic z | Stochastic z ~ N(μ, σ²) |
| **Sampling** | No sampling | Reparameterization trick |
| **Loss Function** | Reconstruction only | Reconstruction + KL divergence |
| **Latent Space** | Irregular, gaps | Smooth, continuous |
| **Generation** | Poor (random points fail) | Good (sample from N(0, I)) |
| **Probabilistic** | No | Yes |
| **Use Case** | Compression, denoising | Generation, sampling |

**Key Insight:**

The KL divergence term forces the latent space to be smooth and continuous, making it possible to:
- Sample random points and get meaningful outputs
- Interpolate smoothly between points
- Perform latent space arithmetic

## 2. Setup & Utilities

Let's import libraries and create helper functions for VAE visualization.

In [ ]:
# Core libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

# Data and visualization
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

# Utilities
import time
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("\nSeeds set for reproducibility")

In [ ]:
# Helper functions for VAE

def visualize_samples(images, title="Generated Samples", n=10):
    """Display a grid of images."""
    fig, axes = plt.subplots(1, n, figsize=(20, 2))
    for i in range(n):
        axes[i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[i].axis('off')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_latent_space(model, data_loader, device, labels_list=None, title="Latent Space"):
    """Visualize 2D latent space with class labels."""
    model.eval()
    latent_vectors = []
    labels = []
    
    with torch.no_grad():
        for images, lbls in data_loader:
            images = images.view(images.size(0), -1).to(device)
            encoded = model.encode(images)
            mu = encoded[0] if isinstance(encoded, (tuple, list)) else encoded
            latent_vectors.append(mu.cpu().numpy())
            labels.append(lbls.numpy())
    
    latent_vectors = np.concatenate(latent_vectors, axis=0)
    labels = np.concatenate(labels, axis=0)
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(latent_vectors[:, 0], latent_vectors[:, 1], 
                         c=labels, cmap='tab10', alpha=0.6, s=5)
    plt.colorbar(scatter, label='Class')
    plt.xlabel('Latent Dimension 1', fontsize=12)
    plt.ylabel('Latent Dimension 2', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_vae_losses(recon_losses, kl_losses, total_losses, title="VAE Training"):
    """Plot reconstruction, KL, and total losses."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    epochs = range(1, len(total_losses) + 1)
    
    axes[0].plot(epochs, recon_losses, 'b-', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Reconstruction Loss')
    axes[0].set_title('Reconstruction Loss')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs, kl_losses, 'r-', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('KL Divergence')
    axes[1].set_title('KL Divergence')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(epochs, total_losses, 'g-', linewidth=2)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Total Loss')
    axes[2].set_title('Total Loss (Recon + KL)')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def sample_from_prior(model, n_samples=10, device='cpu'):
    """Generate new samples by sampling from N(0, I)."""
    model.eval()
    with torch.no_grad():
        # Sample from standard normal
        z = torch.randn(n_samples, model.latent_dim).to(device)
        # Decode
        samples = model.decode(z)
        samples = samples.view(-1, 1, 28, 28)
    return samples

## 3. Load Fashion-MNIST Dataset

We'll use Fashion-MNIST throughout this workshop for more interesting visual results.

In [ ]:
# Hyperparameters
batch_size = 128

# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load Fashion-MNIST
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Class names
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {class_names}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 10, figsize=(20, 4))

for i in range(10):
    img, label = train_dataset[i]
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'{class_names[label]}', fontsize=9)
    axes[0, i].axis('off')
    
    img2, label2 = train_dataset[i+10]
    axes[1, i].imshow(img2.squeeze(), cmap='gray')
    axes[1, i].set_title(f'{class_names[label2]}', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Fashion-MNIST Dataset Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Baseline: Standard Autoencoder

Before implementing VAE, let's quickly train a standard autoencoder for comparison.

This will help us see the **limitations** of deterministic autoencoders.

In [ ]:
class StandardAutoencoder(nn.Module):
    """Standard deterministic autoencoder for comparison."""
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=2):
        super(StandardAutoencoder, self).__init__()
        
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)


# Initialize
baseline_model = StandardAutoencoder(latent_dim=2).to(device)
print(baseline_model)
print(f"\nLatent dimension: 2 (for visualization)")

In [ ]:
# Train baseline autoencoder (quick training)
baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)
baseline_criterion = nn.BCELoss()

num_epochs_baseline = 10
baseline_losses = []

print("Training baseline autoencoder...")

for epoch in range(num_epochs_baseline):
    baseline_model.train()
    train_loss = 0.0
    
    for images, _ in train_loader:
        images = images.view(images.size(0), -1).to(device)
        
        recon = baseline_model(images)
        loss = baseline_criterion(recon, images)
        
        baseline_optimizer.zero_grad()
        loss.backward()
        baseline_optimizer.step()
        
        train_loss += loss.item()
    
    avg_loss = train_loss / len(train_loader)
    baseline_losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_baseline}] Loss: {avg_loss:.6f}")

print("Baseline training completed!")

### Baseline Results & Limitations

Let's visualize the latent space and try to generate samples.

In [ ]:
# Visualize baseline latent space
plot_latent_space(baseline_model, test_loader, device, title="Standard Autoencoder - Latent Space")

print("Observation: Notice the latent space has gaps and irregular structure.")

## 5. VAE Implementation

Now let's implement the Variational Autoencoder with all the components we discussed.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=2):
        super(VAE, self).__init__()
        
        self.latent_dim = latent_dim
        
        # Encoder layers
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)      # Mean μ
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)  # Log-variance log(σ²)
        
        # Decoder layers
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
    
    def encode(self, x):
        """
        Encode input to latent parameters.
        
        Args:
            x: Input tensor (batch_size, input_dim)
        
        Returns:
            mu: Mean of latent distribution (batch_size, latent_dim)
            log_var: Log-variance of latent distribution (batch_size, latent_dim)
        """
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        log_var = self.fc_logvar(h)
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        """
        Reparameterization trick: z = μ + σ * ε
        
        Args:
            mu: Mean (batch_size, latent_dim)
            log_var: Log-variance (batch_size, latent_dim)
        
        Returns:
            z: Sampled latent vector (batch_size, latent_dim)
        """
        std = torch.exp(0.5 * log_var)  # σ = exp(0.5 * log(σ²))
        eps = torch.randn_like(std)      # ε ~ N(0, 1)
        z = mu + std * eps               # z = μ + σ * ε
        return z
    
    def decode(self, z):
        """
        Decode latent vector to reconstruction.
        
        Args:
            z: Latent vector (batch_size, latent_dim)
        
        Returns:
            recon: Reconstructed output (batch_size, input_dim)
        """
        h = F.relu(self.fc3(z))
        recon = torch.sigmoid(self.fc4(h))
        return recon
    
    def forward(self, x):
        """
        Full forward pass through VAE.
        
        Args:
            x: Input tensor (batch_size, input_dim)
        
        Returns:
            recon: Reconstruction (batch_size, input_dim)
            mu: Mean of latent distribution
            log_var: Log-variance of latent distribution
        """
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon = self.decode(z)
        return recon, mu, log_var


# Initialize VAE
vae_model = VAE(latent_dim=2).to(device)

total_params = sum(p.numel() for p in vae_model.parameters())
print(vae_model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Latent dimension: 2")

### Understanding the Architecture

**Key Differences from Standard Autoencoder:**

1. **Encoder outputs TWO vectors**: μ and log(σ²)
   - μ: mean of the latent distribution
   - log(σ²): log-variance (for numerical stability)

2. **Reparameterization layer**: Samples z using the trick
   - z = μ + σ * ε where ε ~ N(0, 1)
   - Allows gradients to flow through

3. **Probabilistic interpretation**: 
   - Each input maps to a distribution, not a point
   - Enables smooth latent space

## 6. VAE Loss Function

The VAE loss combines reconstruction and KL divergence.

In [ ]:
def vae_loss_function(recon_x, x, mu, log_var, beta=1.0):
    """
    VAE loss = Reconstruction Loss + β * KL Divergence
    
    Args:
        recon_x: Reconstructed input
        x: Original input
        mu: Mean of latent distribution
        log_var: Log-variance of latent distribution
        beta: Weight for KL term (default=1.0)
    
    Returns:
        total_loss: Combined loss
        recon_loss: Reconstruction loss component
        kl_loss: KL divergence component
    """
    # Reconstruction loss (Binary Cross-Entropy)
    # Sum over all pixels, then average over batch
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum') / x.size(0)
    
    # KL Divergence: -0.5 * Σ(1 + log(σ²) - μ² - σ²)
    # Closed form for KL(N(μ, σ²) || N(0, 1))
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    
    # Total loss
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss


# Test the loss function
print("Loss function defined!")
print("\nComponents:")
print("  1. Reconstruction Loss (BCE): Measures reconstruction quality")
print("  2. KL Divergence: Regularizes latent space to be N(0, I)")
print("  3. Beta: Controls trade-off (beta=1.0 is standard VAE)")

### TASK: Try with nn.KLDivLoss()

## 7. Training the VAE

Let's train the VAE and track all three loss components.

In [ ]:
# Training configuration
num_epochs = 20
learning_rate = 0.001
beta = 1.0  # Standard VAE

# Optimizer
vae_optimizer = optim.Adam(vae_model.parameters(), lr=learning_rate)

# Loss tracking
train_total_losses = []
train_recon_losses = []
train_kl_losses = []

print(f"Training VAE for {num_epochs} epochs...")
print(f"Learning rate: {learning_rate}")
print(f"Beta (KL weight): {beta}")

start_time = time.time()

for epoch in range(num_epochs):
    vae_model.train()
    total_loss_epoch = 0.0
    recon_loss_epoch = 0.0
    kl_loss_epoch = 0.0
    
    for batch_idx, (images, _) in enumerate(train_loader):
        images = images.view(images.size(0), -1).to(device)
        
        # Forward pass
        recon, mu, log_var = vae_model(images)
        
        # Compute loss
        total_loss, recon_loss, kl_loss = vae_loss_function(
            recon, images, mu, log_var, beta=beta
        )
        
        # Backward pass
        vae_optimizer.zero_grad()
        total_loss.backward()
        vae_optimizer.step()
        
        # Accumulate losses
        total_loss_epoch += total_loss.item()
        recon_loss_epoch += recon_loss.item()
        kl_loss_epoch += kl_loss.item()
    
    # Average losses
    avg_total = total_loss_epoch / len(train_loader)
    avg_recon = recon_loss_epoch / len(train_loader)
    avg_kl = kl_loss_epoch / len(train_loader)
    
    train_total_losses.append(avg_total)
    train_recon_losses.append(avg_recon)
    train_kl_losses.append(avg_kl)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Total: {avg_total:.4f} | "
              f"Recon: {avg_recon:.4f} | "
              f"KL: {avg_kl:.4f}")

elapsed_time = time.time() - start_time
print(f"Training completed in {elapsed_time:.2f} seconds")
print(f"Final Total Loss: {train_total_losses[-1]:.4f}")
print(f"Final Recon Loss: {train_recon_losses[-1]:.4f}")
print(f"Final KL Loss: {train_kl_losses[-1]:.4f}")

### Training Progress Visualization

In [ ]:
plot_vae_losses(train_recon_losses, train_kl_losses, train_total_losses, 
                title="VAE Training Progress")

print("\nObservations:")
print("- Reconstruction loss decreases (better reconstructions)")
print("- KL divergence stabilizes (latent space regularized)")
print("- Total loss is the sum of both components")

## 8. Latent Space Analysis

Let's explore the latent space and compare it with the standard autoencoder.

In [ ]:
# Visualize VAE latent space
plot_latent_space(vae_model, test_loader, device, title="VAE - Latent Space")

print("- Smoother and more continuous than standard autoencoder")
print("- Classes form more organized clusters")
print("- Fewer gaps - can sample from anywhere")

### Latent Space Grid Visualization

Let's sample a grid of points in latent space and decode them to see what the VAE has learned.

In [ ]:
# Create a grid in latent space and decode
vae_model.eval()

n = 20  # Grid size
grid_range = 3  # Range of latent space to explore

# Create grid
x_values = np.linspace(-grid_range, grid_range, n)
y_values = np.linspace(-grid_range, grid_range, n)

# Generate images for each grid point
grid_images = np.zeros((n, n, 28, 28))

with torch.no_grad():
    for i, x in enumerate(x_values):
        for j, y in enumerate(y_values):
            z = torch.tensor([[x, y]], dtype=torch.float32).to(device)
            decoded = vae_model.decode(z)
            grid_images[i, j] = decoded.cpu().view(28, 28).numpy()

# Plot the grid
fig, ax = plt.subplots(figsize=(12, 12))   # Combine all images into one large image
full_image = np.zeros((n * 28, n * 28))
for i in range(n):
    for j in range(n):
        full_image[i*28:(i+1)*28, j*28:(j+1)*28] = grid_images[i, j]

ax.imshow(full_image, cmap='gray')
# ax.axis('off')
plt.title('Latent Space Grid Visualization\n(Each cell is decoded from a point in 2D latent space)', 
          fontsize=14)
plt.tight_layout()
plt.show()

print("\nObservation: Smooth transitions between different types of clothing!")
print("This shows the latent space is continuous and well-structured.")

## 9. Generation & Sampling

One of the key advantages of VAEs: we can generate NEW samples!

### 9.1 Random Generation (Brand new image)

Sample random points from N(0, I) and decode them.

In [ ]:
# Generate random samples
n_samples = 10
random_samples = sample_from_prior(vae_model, n_samples=n_samples, device=device)

visualize_samples(random_samples, title="Randomly Generated Samples (from N(0, I))", n=n_samples)

print("These are BRAND NEW images, not from the training set!")
print("The VAE learned the distribution and can generate novel samples.")

### 9.2 Latent Space Interpolation

Smoothly interpolate between two images in latent space.

In [ ]:
# Select two test images
vae_model.eval()

idx1, idx2 = 42, 88 # index of test dataset

img1, label1 = test_dataset[idx1] # Dress
img2, label2 = test_dataset[idx2] # Top

print(f"Image 1: {class_names[label1]}") 
print(f"Image 2: {class_names[label2]}")

with torch.no_grad():
    # Encode both images
    x1 = img1.view(1, -1).to(device)
    x2 = img2.view(1, -1).to(device)
    
    mu1, _ = vae_model.encode(x1)
    mu2, _ = vae_model.encode(x2)
    
    # Interpolate
    n_steps = 10
    alphas = np.linspace(0, 1, n_steps)
    
    interpolated = []
    for alpha in alphas:
        z_interp = (1 - alpha) * mu1 + alpha * mu2
        decoded = vae_model.decode(z_interp)
        interpolated.append(decoded.view(28, 28).cpu().numpy())
    
    # Visualize
    fig, axes = plt.subplots(1, n_steps, figsize=(20, 2))
    for i, img in enumerate(interpolated):
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f'α={alphas[i]:.1f}', fontsize=9)
    
    plt.suptitle(f'Latent Space Interpolation: {class_names[label1]} → {class_names[label2]}', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Observation: Smooth, meaningful transitions!")
print("This is possible because VAE creates a continuous latent space.")

In [ ]:
# Pass a new image to trained model: TRY

from PIL import Image
def preprocess_new_image(image_path):
    img = Image.open(image_path)
    
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
    ])
    
    img_tensor = transform(img)
    
    # IMPORTANT: Fashion-MNIST has black backgrounds. 
    img_tensor = 1.0 - img_tensor 
    
    return img_tensor.to(device)

custom_bag = preprocess_new_image('img/images_e.jpg')
vae_model.eval()
with torch.no_grad():
    reconstructed, mu, log_var = vae_model(custom_bag.view(1, -1))

# Plot the result
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(custom_bag.cpu().squeeze(), cmap='gray')
plt.title("New Image (28x28)")
plt.subplot(1, 2, 2)
plt.imshow(reconstructed.cpu().view(28, 28), cmap='gray')
plt.title("VAE's Interpretation")
plt.show()

## 10. Comparison: Standard AE vs VAE

Let's directly compare the two approaches.

In [ ]:
# Side-by-side latent space comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Get latent representations
baseline_model.eval()
vae_model.eval()

baseline_latents = []
vae_latents = []
labels_list = []

with torch.no_grad():
    for images, labels in test_loader:
        images_flat = images.view(images.size(0), -1).to(device)
        
        # Standard AE
        z_baseline = baseline_model.encode(images_flat)
        baseline_latents.append(z_baseline.cpu().numpy())
        
        # VAE (use mean)
        mu_vae, _ = vae_model.encode(images_flat)
        vae_latents.append(mu_vae.cpu().numpy())
        
        labels_list.append(labels.numpy())

baseline_latents = np.concatenate(baseline_latents, axis=0)
vae_latents = np.concatenate(vae_latents, axis=0)
labels_array = np.concatenate(labels_list, axis=0)

# Plot Standard AE
scatter1 = axes[0].scatter(baseline_latents[:, 0], baseline_latents[:, 1], 
                           c=labels_array, cmap='tab10', alpha=0.6, s=5)
axes[0].set_xlabel('Latent Dim 1')
axes[0].set_ylabel('Latent Dim 2')
axes[0].set_title('Standard Autoencoder', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot VAE
scatter2 = axes[1].scatter(vae_latents[:, 0], vae_latents[:, 1], 
                           c=labels_array, cmap='tab10', alpha=0.6, s=5)
axes[1].set_xlabel('Latent Dim 1')
axes[1].set_ylabel('Latent Dim 2')
axes[1].set_title('Variational Autoencoder', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Latent Space Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Differences:")
print("- Standard AE: Irregular, with gaps and clusters")
print("- VAE: Smoother, more continuous, better organized")
print("- VAE's KL regularization creates this smooth structure")

In [ ]:
# Summary table
import pandas as pd

comparison_data = {
    'Aspect': ['Latent Space', 'Generation Quality', 'Regularization', 
               'Probabilistic', 'Use Case'],
    'Standard AE': ['Irregular, gaps', 'Poor', 'None', 'No', 
                    'Compression, denoising'],
    'VAE': ['Smooth, continuous', 'Good', 'KL divergence', 'Yes', 
            'Generation, sampling']
}

df = pd.DataFrame(comparison_data)
print("COMPARISON SUMMARY")
print(df.to_string(index=False))

## 11. Tasks & Exercises

Try these exercises for better understanding!

### Exercise 1: Modify Latent Dimension
**Task**: Change `latent_dim` from 2 to 10 and retrain the VAE.
**Questions**:
- How does reconstruction quality change?
- Can you still visualize the latent space? (Hint: use t-SNE to reduce to 2D)
- What happens to the KL divergence?
```python
vae_model = VAE(latent_dim=10).to(device)
```

### Exercise 2: Experiment with Beta (β-VAE)
**Task**: Train VAE with different beta values: 0.5, 1.0, 2.0

**Questions**:
- Which beta gives better reconstructions?
- Which beta gives better generation?
- What is the trade-off?
```python
beta = 0.5  # Try 0.5, 1.0, 2.0
```

### Exercise 3: Add Convolutional Layers
**Task**: Replace fully-connected layers with Conv2d/ConvTranspose2d

**Hint**: Follow the pattern from the autoencoder workshop

**Expected outcome**: Better image quality

### Exercise 4: Plan Conditional VAE
**Task**: Modify the encoder to also take class labels as input

**Hint**: 
- Concatenate one-hot encoded label with image
- This allows generating specific classes on demand

**Advanced**: Can you generate "a red dress" by conditioning on both class and attributes?


### Exercise 5: Different Datasets
**Task**: Apply the same VAE to MNIST or CIFAR-10

**Modifications needed**:
- MNIST: input_dim = 784 (same as Fashion-MNIST)
- CIFAR-10: input_dim = 3072 (32×32×3), may need CNN

## 12. Summary & Key Takeaways

**1. VAE Theory**
- VAEs map inputs to a **distribution** (μ, σ²) rather than a fixed point.
- The **Reparameterization Trick** enables backpropagation through stochastic sampling.
- The **ELBO** objective balances reconstruction accuracy and latent space regularity.

**2. Architecture Differences**
- **Standard AE**: Deterministic, irregular latent space, poor for generation.
- **VAE**: Probabilistic, smooth latent space, excellent for generation and sampling.

**3. Probabilistic Regularization**
- **KL Divergence** forces the latent distribution toward a standard normal prior $N(0, I)$.
- This prevents the model from "cheating" by mapping each point to a unique, distant location in space.

**4. Generation Capabilities**
- We can sample directly from the latent prior to generate **brand new** samples.
- The manifold structure allows for **linear interpolation**, creating smooth transitions between classes.

### Design Considerations

- **Latent Dimension**: Larger dims improve reconstruction but are harder to visualize and sample.
- **Beta Parameter (β-VAE)**: Adjusting the KL weight allows you to prioritize either reconstruction quality or latent disentanglement.
- **Loss Choice**: BCE is common for pixel values in [0, 1], while MSE is used for continuous data.

### Finally

Variational Autoencoders are a bridge between standard neural networks and deep generative modeling. They provide a principled way to represent complex data in a structured, probabilistic way.